# N-BEATS — Walmart Sales Forecasting

In [ ]:
import subprocess, os, warnings
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pickle
import wandb

warnings.filterwarnings('ignore')

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LEN  = 52
PRED_LEN = 39
WANDB_PROJECT = 'walmart-sales-forecasting-project'
WANDB_ENTITY  = 'ashos22-free-university-of-tbilisi-'

def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5.0, 1.0)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

print(f'Device: {DEVICE}')

## WandB შესვლა

In [ ]:
subprocess.run(['rm', '-f', '/root/.netrc'], capture_output=True)
os.environ['WANDB_API_KEY'] = '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'
wandb.login(key=os.environ['WANDB_API_KEY'], relogin=True)
print('WandB login OK')

## მონაცემების ჩატვირთვა

In [ ]:
DATA_PATH = '/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/'

train    = pd.read_csv(DATA_PATH + 'train.csv.zip')
test     = pd.read_csv(DATA_PATH + 'test.csv.zip')
stores   = pd.read_csv(DATA_PATH + 'stores.csv')
features = pd.read_csv(DATA_PATH + 'features.csv.zip')

print('train:   ', train.shape)
print('test:    ', test.shape)
print('stores:  ', stores.shape)
print('features:', features.shape)
train.head()

## მონაცემების გასუფთავება

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY,
    name='NBEATS_Cleaning', group='NBEATS_Training', reinit=True
)

train_df = train.merge(stores, on='Store')
train_df = train_df.merge(features, on=['Store', 'Date'], suffixes=('', '_feat'))
train_df.drop(columns=['IsHoliday_feat'], inplace=True)

test_df = test.merge(stores, on='Store')
test_df = test_df.merge(features, on=['Store', 'Date'], suffixes=('', '_feat'))
test_df.drop(columns=['IsHoliday_feat'], inplace=True)

for col in ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']:
    train_df[col] = train_df[col].fillna(0)
    test_df[col]  = test_df[col].fillna(0)

for col in ['CPI', 'Unemployment']:
    train_df[col] = train_df[col].ffill()
    test_df[col]  = test_df[col].ffill()

type_map = {'A': 0, 'B': 1, 'C': 2}
train_df['Type'] = train_df['Type'].map(type_map)
test_df['Type']  = test_df['Type'].map(type_map)

train_df['Date'] = pd.to_datetime(train_df['Date'])
test_df['Date']  = pd.to_datetime(test_df['Date'])

wandb.log({
    'train_rows': len(train_df),
    'test_rows':  len(test_df),
    'null_train': int(train_df.isnull().sum().sum())
})
run.finish()
print(f'train_rows: {len(train_df)},  null: {train_df.isnull().sum().sum()}')

## Feature Engineering — სერიების მომზადება

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY,
    name='NBEATS_Feature_Engineering', group='NBEATS_Training', reinit=True
)

series_info = {}
for (store, dept), grp in train_df.groupby(['Store', 'Dept']):
    grp  = grp.sort_values('Date')
    sales = grp['Weekly_Sales'].values.astype(np.float32)
    if len(sales) < SEQ_LEN + PRED_LEN:
        continue
    is_holiday = grp['IsHoliday'].values.astype(bool)
    mean = sales.mean()
    std  = sales.std() + 1e-8
    series_info[(store, dept)] = {
        'sales':      sales,
        'is_holiday': is_holiday,
        'mean':       mean,
        'std':        std,
        'dates':      grp['Date'].values
    }

test_dates   = sorted(test_df['Date'].unique())
n_test_dates = len(test_dates)

wandb.config.update({
    'n_series':       len(series_info),
    'avg_series_len': float(np.mean([len(v['sales']) for v in series_info.values()])),
    'seq_len':        SEQ_LEN,
    'pred_len':       PRED_LEN,
    'n_test_dates':   n_test_dates
})
run.finish()
print(f'n_series: {len(series_info)},  test_dates: {n_test_dates}')

## N-BEATS მოდელის კლასები

In [ ]:
class NBeatsBlock(nn.Module):
    def __init__(self, seq_len, pred_len, layer_width=256):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(seq_len, layer_width),     nn.ReLU(),
            nn.Linear(layer_width, layer_width), nn.ReLU(),
            nn.Linear(layer_width, layer_width), nn.ReLU(),
            nn.Linear(layer_width, layer_width), nn.ReLU(),
        )
        self.backcast_head = nn.Linear(layer_width, seq_len)
        self.forecast_head = nn.Linear(layer_width, pred_len)

    def forward(self, x):
        h = self.fc(x)
        return self.backcast_head(h), self.forecast_head(h)


class NBEATSModel(nn.Module):
    def __init__(self, seq_len, pred_len, n_stacks=2, n_blocks=3, layer_width=256):
        super().__init__()
        self.seq_len  = seq_len
        self.pred_len = pred_len
        self.blocks   = nn.ModuleList([
            NBeatsBlock(seq_len, pred_len, layer_width)
            for _ in range(n_stacks * n_blocks)
        ])

    def forward(self, x):
        residual = x
        forecast  = torch.zeros(x.shape[0], self.pred_len, device=x.device)
        for block in self.blocks:
            backcast, fc = block(residual)
            residual  = residual - backcast
            forecast  = forecast  + fc
        return forecast


print('N-BEATS კლასები მზადაა')

## Dataset და WMAE გამოთვლა

In [ ]:
class WalmartDataset(Dataset):
    def __init__(self, series_info, seq_len, pred_len, split='train'):
        self.X, self.y = [], []
        for key, info in series_info.items():
            sales = info['sales']
            n     = len(sales)
            if n < seq_len + 2 * pred_len:
                continue
            mean, std = info['mean'], info['std']
            norm      = (sales - mean) / std
            val_start = n - pred_len
            if split == 'train':
                for i in range(seq_len, val_start - pred_len + 1):
                    self.X.append(norm[i - seq_len:i])
                    self.y.append(norm[i:i + pred_len])
            else:
                if val_start >= seq_len:
                    self.X.append(norm[val_start - seq_len:val_start])
                    self.y.append(norm[val_start:n])
        self.X = torch.FloatTensor(np.array(self.X))
        self.y = torch.FloatTensor(np.array(self.y))

    def __len__(self):            return len(self.X)
    def __getitem__(self, idx):   return self.X[idx], self.y[idx]


def compute_val_wmae(model, series_info, seq_len, pred_len):
    model.eval()
    all_true, all_pred, all_hol = [], [], []
    with torch.no_grad():
        for key, info in series_info.items():
            sales, is_holiday = info['sales'], info['is_holiday']
            mean, std = info['mean'], info['std']
            n = len(sales)
            val_start = n - pred_len
            if val_start < seq_len:
                continue
            norm = (sales - mean) / std
            x    = torch.FloatTensor(norm[val_start - seq_len:val_start]).unsqueeze(0).to(DEVICE)
            pred = model(x).squeeze().cpu().numpy()
            pred = np.maximum(pred * std + mean, 0)
            all_pred.extend(pred)
            all_true.extend(sales[val_start:val_start + pred_len])
            all_hol.extend(is_holiday[val_start:val_start + pred_len])
    return wmae(np.array(all_true), np.array(all_pred), np.array(all_hol))


print('Dataset კლასი მზადაა')

## ტრენინგის ფუნქცია

In [ ]:
def train_nbeats(series_info, config, run_name):
    run = wandb.init(
        project=WANDB_PROJECT, entity=WANDB_ENTITY,
        name=run_name, group='NBEATS_Training',
        config=config, reinit=True
    )

    model = NBEATSModel(
        seq_len=SEQ_LEN, pred_len=PRED_LEN,
        n_stacks=config['n_stacks'],
        n_blocks=config['n_blocks'],
        layer_width=config['layer_width']
    ).to(DEVICE)

    train_ds     = WalmartDataset(series_info, SEQ_LEN, PRED_LEN, 'train')
    val_ds       = WalmartDataset(series_info, SEQ_LEN, PRED_LEN, 'val')
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=config['batch_size'])

    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    criterion = nn.MSELoss()

    best_val_loss    = float('inf')
    best_state       = None
    patience_counter = 0

    for epoch in range(config['epochs']):
        model.train()
        train_loss = 0.0
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                val_loss += criterion(model(X), y).item()
        val_loss /= len(val_loader)

        wandb.log({'epoch': epoch + 1, 'train_loss': train_loss, 'val_loss': val_loss})

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_state       = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config['patience']:
                break

    model.load_state_dict(best_state)
    wmae_val = compute_val_wmae(model, series_info, SEQ_LEN, PRED_LEN)
    wandb.log({'wmae_val': wmae_val, 'best_val_loss': best_val_loss})
    run.finish()
    return model, wmae_val


print('train_nbeats() მზადაა')

## Baseline ტრენინგი

In [ ]:
baseline_config = {
    'n_stacks':    2,
    'n_blocks':    3,
    'layer_width': 256,
    'lr':          0.001,
    'batch_size':  256,
    'epochs':      50,
    'patience':    10
}

baseline_model, baseline_wmae = train_nbeats(series_info, baseline_config, 'NBEATS_Baseline')
print(f'Baseline WMAE: {baseline_wmae:.4f}')

## Tuned ტრენინგი

In [ ]:
tuned_config = {
    'n_stacks':    3,
    'n_blocks':    3,
    'layer_width': 256,
    'lr':          0.0005,
    'batch_size':  128,
    'epochs':      60,
    'patience':    12
}

tuned_model, tuned_wmae = train_nbeats(series_info, tuned_config, 'NBEATS_Tuned')
print(f'Tuned WMAE: {tuned_wmae:.4f}')

## შედარება — საუკეთესო მოდელი

In [ ]:
scores = {
    'NBEATS_Baseline': (baseline_model, baseline_wmae, baseline_config),
    'NBEATS_Tuned':    (tuned_model,    tuned_wmae,    tuned_config)
}

best_name  = min(scores, key=lambda k: scores[k][1])
best_model, best_wmae, best_config = scores[best_name]

print(f'NBEATS_Baseline WMAE: {baseline_wmae:.4f}')
print(f'NBEATS_Tuned    WMAE: {tuned_wmae:.4f}')
print(f'საუკეთესო: {best_name}  →  WMAE {best_wmae:.4f}')

## Test Predictions და Submission

In [ ]:
def predict_test(model, series_info, test_df, seq_len):
    model.eval()
    result = test_df.copy()
    result['Date'] = pd.to_datetime(result['Date'])
    result['Weekly_Sales'] = 0.0

    with torch.no_grad():
        for (store, dept), info in series_info.items():
            mask = (result['Store'] == store) & (result['Dept'] == dept)
            if mask.sum() == 0:
                continue
            sales      = info['sales']
            mean, std  = info['mean'], info['std']
            if len(sales) < seq_len:
                continue
            norm = (sales - mean) / std
            x    = torch.FloatTensor(norm[-seq_len:]).unsqueeze(0).to(DEVICE)
            pred = model(x).squeeze().cpu().numpy()
            pred = np.maximum(pred * std + mean, 0)

            dates_sorted = sorted(result.loc[mask, 'Date'].unique())
            date_to_idx  = {d: i for i, d in enumerate(dates_sorted)}
            for row_idx, row in result.loc[mask].iterrows():
                idx = date_to_idx.get(row['Date'])
                if idx is not None and idx < len(pred):
                    result.loc[row_idx, 'Weekly_Sales'] = pred[idx]

    return result


predictions_df = predict_test(best_model, series_info, test_df, SEQ_LEN)

submission = predictions_df[['Store', 'Dept', 'Date', 'Weekly_Sales']].copy()
submission['Date'] = submission['Date'].dt.strftime('%Y-%m-%d')
submission['Id']   = submission.apply(
    lambda r: f"{int(r['Store'])}_{int(r['Dept'])}_{r['Date']}", axis=1
)
submission = submission[['Id', 'Weekly_Sales']]
submission.to_csv('nbeats_submission.csv', index=False)

print(f'Submission shape: {submission.shape}')
submission.head()

## WandB Artifact — საუკეთესო მოდელის შენახვა

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY,
    name='NBEATS_Best_Pipeline', group='NBEATS_Training', reinit=True
)

torch.save(best_model.state_dict(), 'nbeats_model.pt')

with open('series_info.pkl', 'wb') as f:
    pickle.dump(series_info, f)

with open('nbeats_config.pkl', 'wb') as f:
    pickle.dump({
        'seq_len':     SEQ_LEN,
        'pred_len':    PRED_LEN,
        'n_stacks':    best_config['n_stacks'],
        'n_blocks':    best_config['n_blocks'],
        'layer_width': best_config['layer_width']
    }, f)

artifact = wandb.Artifact(
    name='nbeats-walmart-sales',
    type='model',
    metadata={'wmae_val': best_wmae, 'best_version': best_name}
)
artifact.add_file('nbeats_model.pt')
artifact.add_file('series_info.pkl')
artifact.add_file('nbeats_config.pkl')
run.log_artifact(artifact)

wandb.log({'wmae_val_best': best_wmae})
run.finish()

print(f'WandB Artifact: nbeats-walmart-sales')
print(f'საუკეთესო WMAE: {best_wmae:.4f} ({best_name})')